In [ ]:
import squidpy as sq
import scanpy as sc
import scipy as scp
import numpy as np 
import pandas as pd
import scvi
import seaborn as sns
import pyensembl as ensb
import matplotlib as plt
import liana as li

In [ ]:
control = sc.read_h5ad('data/normalCellsabv55.h5ad')
sclc = sc.read_h5ad('data/sclcCells.h5ad')

In [ ]:
def qualityControl(adata):
    '''
    Compute quality control metrics: label mitochondrial and ribosomal genes and run sc.calculate_qc_metrics
    '''
    adata.var['mitochondrial'] = adata.var.index.str.startswith('MT-')
    ribosomalGeneSet = "http://software.broadinstitute.org/gsea/msigdb/download_geneset.jsp?geneSetName=KEGG_RIBOSOME&fileType=txt" # filter out ribosomal genes too
    df_ribo = pd.read_table(ribosomalGeneSet, skiprows=2, header = None)
    adata.var[adata.var.index.isin(df_ribo[0].values)] 
    adata.var['ribosomal'] = adata.var.index.isin(df_ribo[0].values) 
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mitochondrial', 'ribosomal'], percent_top=None, log1p=False, inplace=True)
    sc.pp.filter_genes(adata, min_cells=3) # filter genes in a minimum of 3 cells
    sc.pp.filter_cells(adata, min_genes=200) # filter cells with a minimum of 200 genes
    return adata

def normalization(adata):
    '''
    Perform normalization to unify counts
    '''
    sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10,000 UMI
    sc.pp.log1p(adata) #change to log counts
    adata.X.sum(axis = 1)
    adata.raw = adata

def ensemblIDfGene(adata,gene):
    '''
    ENSEMBL ID from gene
    '''
    obj = adata.var[adata.var.feature_name == gene]
    id = obj.index.tolist()[0]
    return id

def genefEnsemblID(adata,id):
    '''
    Gene from ENSEMBL ID
    '''
    obj = adata.var[adata.var.index == id]
    gene_name = obj.feature_name.tolist()[0]
    return gene_name


In [ ]:
qualityControl(sclc) # AnnData object with n_obs × n_vars = 20577 × 21199
qualityControl(control) # AnnData object with n_obs × n_vars = 61516 × 25545

In [ ]:
sc.pl.violin(control, [
    'n_genes_by_counts', 
    'total_counts', 
    'pct_counts_mitochondrial', 
    'pct_counts_ribosomal'
    ], 
    jitter=0.4, 
    multi_panel=True)

In [ ]:
sc.pl.violin(sclc, [
    'n_genes_by_counts', 
    'total_counts', 
    'pct_counts_mitochondrial', 
    'pct_counts_ribosomal'
    ], 
    jitter=0.4, 
    multi_panel=True)

In [ ]:
# Check metrics
print('Minimum genes in a cell in the SCLC condition data: ',sclc.obs.n_genes_by_counts.min())
print('Avg total count of a gene in the SCLC condition data: ',sclc.obs.total_counts.mean())
print('Minimum genes in a cell in the normal condition data: ',control.obs.n_genes_by_counts.min())
print('Avg total count of a gene in the normal condition data: ',control.obs.total_counts.mean())

In [ ]:
upper_lim = np.quantile(sclc.obs.n_genes_by_counts.values, .98)
upper_lim
sclc = sclc[sclc.obs.n_genes_by_counts < upper_lim]
sclc = sclc[sclc.obs.pct_counts_mitochondrial < 20] # less than 20%
sclc = sclc[sclc.obs.pct_counts_ribosomal < 20] # less than 10%
sclc

In [ ]:
upper_lim_ctrl = np.quantile(control.obs.n_genes_by_counts.values, .98)
upper_lim_ctrl
control = control[control.obs.n_genes_by_counts < upper_lim]
control = control[control.obs.pct_counts_mitochondrial < 20] # less than 20%
control = control[control.obs.pct_counts_ribosomal < 20] # less than 10%
control

In [ ]:
# Normalization
normalization(control)
normalization(sclc)

In [ ]:
sc.pp.highly_variable_genes(sclc, n_top_genes = 2000)
sc.pl.highly_variable_genes(sclc)

In [ ]:
sc.pp.highly_variable_genes(control, n_top_genes = 2000)
sc.pl.highly_variable_genes(control)

In [ ]:
sclc = sclc[:, sclc.var.highly_variable]
sc.pp.regress_out(sclc, ['total_counts', 'pct_counts_mitochondrial', 'pct_counts_ribosomal'])
sc.pp.scale(sclc, max_value=10)
sc.tl.pca(sclc, svd_solver='arpack')
sc.pl.pca_variance_ratio(sclc, log=True, n_pcs = 50)

In [ ]:
control = control[:, control.var.highly_variable]
sc.pp.regress_out(control, ['total_counts', 'pct_counts_mitochondrial', 'pct_counts_ribosomal'])
sc.pp.scale(control, max_value=10)
sc.tl.pca(control, svd_solver='arpack')
sc.pl.pca_variance_ratio(control, log=True, n_pcs = 50)

In [ ]:
sc.pp.neighbors(sclc, n_pcs = 30)
sc.tl.umap(sclc)
sc.pl.umap(sclc)

In [ ]:
sc.pp.neighbors(control, n_pcs = 30)
sc.tl.umap(control)
sc.pl.umap(control)

In [ ]:
sc.tl.leiden(sclc, resolution = 0.5)

sc.pl.umap(sclc, color=['leiden'], legend_loc='on data')
sc.pl.umap(sclc, color=['cell_type'], legend_loc='right margin')

# sc.tl.paga(sclc, groups = ['leiden','cell_type'])

# cluster_labels = sclcCells.obs['leiden']
# cluster_labels

In [ ]:
sc.tl.leiden(control, resolution = 0.5)

sc.pl.umap(control, color=['leiden'], legend_loc='on data')
sc.pl.umap(control, color=['cell_type'], legend_loc='right margin')

In [ ]:

# sc.pl.umap(control, color=['leiden'], legend_loc='on data')
sc.pl.umap(sclc, color='cell_type', groups=['CD4-positive, alpha-beta T cell'], legend_loc='right margin', size=20)
sc.pl.umap(sclc, color=['IL32', 'TRAC', 'CD3D'], legend_loc='right margin', size=20)
sc.pl.umap(sclc, color=['EPCAM', 'LGMN'], legend_loc='right margin', size=20)

In [ ]:
sc.pl.umap(sclc, color='cell_type', groups=['respiratory basal cell'], legend_loc='right margin', size=20)
sc.pl.umap(sclc, color=['TP63'], legend_loc='right margin', size=20)
sc.pl.umap(control, color='cell_type', groups=['respiratory basal cell'], legend_loc='right margin', size=20)
sc.pl.umap(control, color=['TP63'], legend_loc='right margin', size=20)
# sc.pl.umap(control, color=['CXCR4', 'BTG1', 'IL7R'], legend_loc='right margin', size=20)

In [ ]:
sc.pl.umap(sclc, color='cell_type', groups=['respiratory basal cell'], legend_loc='right margin', size=20)
sc.pl.umap(sclc, color=['TP63'], legend_loc='right margin', size=20)
sc.pl.umap(control, color='cell_type', groups=['respiratory basal cell'], legend_loc='right margin', size=20)
sc.pl.umap(control, color=['TP63'], legend_loc='right margin', size=20)

In [ ]:
sc.pl.umap(sclc, color='cell_type', groups=['type I pneumocyte'], legend_loc='right margin', size=20)
sc.pl.umap(sclc, color=['AGER', 'CAV1'], legend_loc='right margin', size=20)
sc.pl.umap(control, color='cell_type', groups=['type I pneumocyte'], legend_loc='right margin', size=20)
sc.pl.umap(control, color=['AGER', 'CAV1'], legend_loc='right margin', size=20)

In [ ]:
myofibroblast = sclc.obs.cell_type == 'myofibroblast cell'
multiciliated = sclc.obs.cell_type == 'multi-ciliated epithelial cell'
sclc = sclc[~myofibroblast, :]
sclc = sclc[~multiciliated, :]
sclc

In [ ]:
sc.tl.rank_genes_groups(sclc, 'cell_type')
sc.pl.rank_genes_groups(sclc, n_genes=20, sharey=False)

In [ ]:

# Get the results of the gene ranking analysis
result_table = pd.DataFrame(sclc.uns['rank_genes_groups']['names']).head(5)

result_table.T

In [ ]:
sc.tl.rank_genes_groups(control, 'cell_type')
sc.pl.rank_genes_groups(control, n_genes=20, sharey=False)

In [ ]:
result_table = pd.DataFrame(control.uns['rank_genes_groups']['names']).head(5)

result_table.T


In [ ]:
def markers(adata):
    '''
    Get marker genes from rank_genes_groups()
    '''
    markers = sc.get.rank_genes_groups_df(adata, None)
    markers = markers[(markers.pvals_adj < 0.05) & (markers.logfoldchanges > 1)]
    return markers

In [ ]:
sclcMarkers = markers(sclc)
sclcMarkers.to_csv('results/sclcMarkers.csv')
controlMarkers = markers(control)
controlMarkers.to_csv('results/controlMarkers.csv')

In [ ]:
sclcMarkers

In [ ]:
controlMarkers


In [ ]:

# cellTypeDict = {
#     '0':'alveolar macrophage',    
#     '1':'lung macrophage',
#     '2':'', # native
#     '3':'CD1c+ myeloid dentritic cell',
#     '4':'capillary endothelial',
#     '5':'T-cell cluster', # CD4+, CD8+ T-cells
#     '6':'vein endothelial cell',
#     '7':'type 1 pneumocyte',
#     '8':'native',
#     '9':'elicited macrophage',
#     '10':'fibroblast', # alveolar type 1 fibroblast, bronchous fibroblast, alveolar type 2 fibroblast
#     '11':'type 2 pneumocyte',
#     '12':'natural killer cell',
#     '13':'NK Cells',
#     '14':'', # native
#     '15':'mast cell',
#     '16':'plasma cell',

# }

In [ ]:
sc.pp.neighbors(sclc, n_neighbors=4, n_pcs=20)
sc.tl.draw_graph(sclc)


In [ ]:
sc.pl.draw_graph(sclc, color="cell_type", legend_loc="on data")


In [ ]:
sc.tl.paga(sclc, groups='cell_type')

with plt.rc_context({"figure.figsize": (30, 30)}):
    sc.pl.paga(sclc, color='cell_type')

In [ ]:
sc.tl.paga(control, groups='cell_type')

with plt.rc_context({"figure.figsize": (25, 25)}):
    sc.pl.paga(control, color='cell_type')

In [ ]:
# Filter data for specific cell types
selected_cell_types = [
    'T cell',
    'B cell',
    'CD4-positive, alpha-beta T cell',
    'CD8-positive, alpha-beta T cell',
    'natural killer cell',
    'classical macrophage',
    'elicited macrophage',
    'lung macrophage',
]  # Add more cell types if needed
subset_sclc = sclc[sclc.obs['cell_type'].isin(selected_cell_types)]

# Run PAGA analysis
sc.pp.neighbors(subset_sclc)
sc.tl.draw_graph(subset_sclc)
sc.tl.paga(subset_sclc, groups='cell_type')

# Plot PAGA graph with manually set colors
sc.pl.paga(subset_sclc, color='cell_type')

In [ ]:
# Filter data for specific cell types
selected_cell_types = [
    'T cell',
'B cell',
'CD4-positive, alpha-beta T cell',
'CD8-positive, alpha-beta T cell',
'natural killer cell',
'classical macrophage',
'elicited macrophage',
'lung macrophage',
'bronchus fibroblast of lung',
'alveolar type 1 fibroblast cell',
'alveolar type 2 fibroblast cell'
]  # Add more cell types if needed
subset_control = control[control.obs['cell_type'].isin(selected_cell_types)]

# Run PAGA analysis
sc.pp.neighbors(subset_control, n_neighbors=4, n_pcs=20)
sc.tl.draw_graph(subset_control)
sc.tl.paga(subset_control, groups='cell_type')

# Plot PAGA graph with manually set colors
sc.pl.paga(subset_control, color='cell_type')

In [ ]:
sc.tl.paga(control, groups='cell_type')

with plt.rc_context({"figure.figsize": (15, 15)}):
    sc.pl.paga(control, color='cell_type', legend_loc='right margin')

In [ ]:
with plt.rc_context({"figure.figsize": (10, 10)}):
    sc.pl.paga(sclc, color=["A2M"])

In [ ]:
%pip install diffxpy

In [ ]:
import diffxpy.api as de

In [ ]:
subset = adata[adata.obs['cell type'].isin(['AT1', 'AT2'])].copy()